# Day 41 · A loss for language: negative log likelihood

**First-principles question:** How do you score a language model with one number?

The full task (learn → build → record) is in this folder's `README.md`. This notebook is your workbench:
run cells top to bottom with **Shift + Enter**, fill in every cell marked **TODO**, and finish with the
**✅ self-check** cell at the end.

In [ ]:
import sys
from pathlib import Path

def find_root():
    """The fde-first-principles folder (it has progress.json)."""
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "progress.json").exists():
            return p
    raise RuntimeError("Open this notebook from inside your fde-first-principles folder.")

ROOT = find_root()
sys.path.insert(0, str(ROOT))  # so `import lab.…` works

import urllib.request
import numpy as np
import matplotlib.pyplot as plt

def download(url, name):
    """Download once into data/downloads/ (ignored by git), then reuse."""
    path = ROOT / "data" / "downloads" / name
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(url, path)
    return path.read_text(encoding="utf-8")

words = download("https://raw.githubusercontent.com/karpathy/makemore/master/names.txt", "names.txt").split()
chars = sorted(set("".join(words)))
stoi = {ch: i + 1 for i, ch in enumerate(chars)}; stoi["."] = 0      # "." marks the start and the end of a name
itos = {i: ch for ch, i in stoi.items()}
print(len(words), "names, e.g.", words[:5])
N = np.zeros((27, 27), dtype=np.int64)
for w in words:
    seq = ["."] + list(w) + ["."]
    for ch1, ch2 in zip(seq, seq[1:]):
        N[stoi[ch1], stoi[ch2]] += 1

## 1 · How surprised is the model by real names?

In [ ]:
def nll(word, P):
    """Negative log likelihood of one name: minus the sum of log P(next | current) over its pairs."""
    seq = ["."] + list(word) + ["."]
    # TODO: add up -np.log(P[stoi[a], stoi[b]]) for every pair (a, b)
    ...

P = N / N.sum(axis=1, keepdims=True)
total = sum(nll(w, P) for w in words)
pairs = sum(len(w) + 1 for w in words)
print(f"average NLL per pair: {total / pairs:.4f}")

## 2 · Your name, and nonsense

In [ ]:
with np.errstate(divide="ignore"):
    for w in ["raj", "andrej", "xqzq"]:          # TODO: add your own name and a friend's
        print(f"{w:8} NLL {nll(w, P):.2f}")

## 3 · Smoothing
'xqzq' gives infinity: some pairs were never seen. Add 1 to every count ("fake counts").

In [ ]:
# TODO: P_smooth = (N + 1) normalised by row
P_smooth = None
print(f"average NLL with smoothing: {sum(nll(w, P_smooth) for w in words) / pairs:.4f}")
print("xqzq:", nll("xqzq", P_smooth))

**What does 'average NLL 2.45' mean, in words?** …

In [ ]:
# ✅ Self-check: run me last. A clear message tells you what to fix.
assert abs(total / pairs - 2.454) < 0.01, "The average NLL of the counting model is about 2.45"
assert np.isfinite(nll("xqzq", P_smooth)), "Smoothing should remove the infinity"
assert np.allclose(P_smooth.sum(axis=1), 1)
print("All checks passed ✅  Now write 'In my words' in README.md and record your video.")